# Reverse Query Demo

Use PxFquery for function-to-perturbation questions. Reverse queries rank perturbation candidates associated with a requested functional state.

In [1]:
from pathlib import Path
import os
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", message="IProgress not found.*")

repo_root = Path.cwd()
if not (repo_root / "src" / "pxfquery").exists() and (repo_root.parent / "src" / "pxfquery").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Optional: load local environment variables for the LLM provider.
for env_file in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.home() / ".env"]:
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

from pxfquery import PxFQuery

pxf = PxFQuery()
print("PxFquery", pxf.version)
resource_status = pxf.resources.status()
resource_info = resource_status.to_dict() if hasattr(resource_status, "to_dict") else dict(resource_status)
print("Resource status:")
print({
    "available": resource_info.get("available"),
    "source": resource_info.get("source"),
    "version": resource_info.get("version"),
    "available_file_count": len(resource_info.get("available_files") or {}),
})


PxFquery 0.5.12.dev0
Resource status:
{'available': True, 'source': 'manifest', 'version': 'v20260628', 'available_file_count': 18}


In [2]:
question = (
    "In a lung adenocarcinoma model, which perturbations are linked to suppression of "
    "inflammatory response and preservation of oxidative phosphorylation?"
)

print("Question:")
print(question)

qdata = pxf.tl.parse(question, top_n=10)
pxf.tl.answer(qdata)
answer = pxf.get.answer(qdata)

print("\nAnswer:")
print(answer)

Question:
In a lung adenocarcinoma model, which perturbations are linked to suppression of inflammatory response and preservation of oxidative phosphorylation?


[Parsing] start


[Parsing] done | mode=reverse; context=lung adenocarcinoma model; time=1.84s
[Matching] start


[Matching] done | matches=6; time=6.09s
[Matrix] start


[Matrix] done | profiles=6; skipped=0; time=19.56s
[Evidence] start



Answer:
Answer
Multiple perturbations are strongly linked to suppression of inflammatory response and preservation of oxidative phosphorylation in lung adenocarcinoma models. Top candidates include AZD-7687, BRD-K08451418, BRD-K28472299, BRD-K46255814, BRD-K46670060, BRD-K46961308, BRD-K47700511, BRD-K94493764, GPR151_GALP, GPR151_M617, NFKBIA, and atenolol, all showing perfect functional match scores in A549 or SKLU1 cells. NFKBIA is a gene knockout candidate, while the others are drug treatments. Additional candidates with strong support include drospirenone and indoprofen in BEN cells, and sulforaphane and erastin in DV90 cells, though with lower scores.

Analysis source: pxfquery 0.5.12.dev0
Evidence: inspect `answer.tables["route_summary"]`, `answer.tables["ranked_results"]`, and `answer.tables["route_target_functions"]`.
Figures: call `pxf.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.


[Evidence] done | status=ready; time=8.67s


In [3]:
print("Candidate ranking preview:")
display(answer.tables.get("ranked_results", answer.biological_results)[:10])


Candidate ranking preview:


[{'label': 'AZD-7687',
  'score': 10.0,
  'mean_score': 10.0,
  'exact_cell_score': None,
  'best_score': 10.0,
  'best_exact_cell_score': None,
  'exact_cell_support': False,
  'support_routes': 1,
  'support_cells': 1,
  'route_ids': 'reverse_003',
  'cells': 'A549',
  'best_route_id': 'reverse_003',
  'best_cell': 'A549',
  'pert_id': 'BRD-K31495718',
  'cmap_name': 'AZD-7687',
  'recommended_operation': 'drug_treat',
  'score_orientation': 'observed_perturbation_effect',
  'kind': 'candidate_consensus',
  'source': 'cp',
  'rank': 1},
 {'label': 'BRD-K08451418',
  'score': 10.0,
  'mean_score': 10.0,
  'exact_cell_score': None,
  'best_score': 10.0,
  'best_exact_cell_score': None,
  'exact_cell_support': False,
  'support_routes': 1,
  'support_cells': 1,
  'route_ids': 'reverse_001',
  'cells': 'SKLU1',
  'best_route_id': 'reverse_001',
  'best_cell': 'SKLU1',
  'pert_id': 'BRD-K08451418',
  'cmap_name': 'BRD-K08451418',
  'recommended_operation': 'drug_treat',
  'score_orientati

In [4]:
print("Target-function evidence preview:")
display(answer.tables.get("route_target_functions", [])[:10])


Target-function evidence preview:


[{'route_id': 'reverse_001',
  'cell': 'SKLU1',
  'modality': 'cp',
  'interpretation_set_id': 'exact',
  'function_rank': 1,
  'label': 'Inflammatory Response',
  'var_name': 'HALLMARK_INFLAMMATORY_RESPONSE',
  'source': 'hallmark',
  'direction': 'suppress',
  'input': 'inflammatory response',
  'target_weight': -1.0,
  'route_quality': 'direct_or_close_representative',
  'cell_match_type': 'concept_representative_cell'},
 {'route_id': 'reverse_002',
  'cell': 'DV90',
  'modality': 'cp',
  'interpretation_set_id': 'exact',
  'function_rank': 1,
  'label': 'Inflammatory Response',
  'var_name': 'HALLMARK_INFLAMMATORY_RESPONSE',
  'source': 'hallmark',
  'direction': 'suppress',
  'input': 'inflammatory response',
  'target_weight': -1.0,
  'route_quality': 'direct_or_close_representative',
  'cell_match_type': 'concept_representative_cell'},
 {'route_id': 'reverse_003',
  'cell': 'A549',
  'modality': 'cp',
  'interpretation_set_id': 'exact',
  'function_rank': 1,
  'label': 'Inflamma